# 07 — Prepare the frozen Bentong SVM for Google Earth Engine deployment

**Purpose:** convert the frozen `SVM + S2 + R1` sklearn model into three
auditable GEE tables:

1. a low-variance weighted-resampling training table;
2. an independent deployment-agreement verification table;
3. a one-row scaler/SVM configuration table.

The resulting GEE classifier is a **deployment replica** of the frozen
sklearn model. It is not a replacement model and is not used to re-select
features or tune hyperparameters.


## Why this conversion is necessary

A sklearn `.joblib` file cannot be executed directly inside Earth Engine.
The frozen model also used continuous per-pixel weights, whereas the GEE
`Classifier.train` interface has no sample-weight argument. This notebook
therefore:

- keeps the frozen 13-band order, weighted scaler, effective RBF gamma,
  class encoding and the C-SVC settings exposed by GEE;
- approximates continuous classifier weights by deterministic systematic
  resampling;
- compensates the resampling multiplier in the GEE `cost` value;
- records sklearn `tol=0.001` as metadata because GEE's documented
  `terminationEpsilon` option is not available for C-SVC;
- prepares held-out raw pixels with frozen-sklearn predictions so that the
  GEE replica must be checked before map export is enabled.

The verification agreement is a **software deployment parity check**, not
a new model-accuracy estimate. The thesis accuracy remains the grouped
validation and external validation already completed.


## 1. Mount Google Drive and load libraries

This cell only mounts Drive and imports packages. It does not retrain or
overwrite the frozen model.


In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


import gc
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)

## 2. Paths and locked deployment settings

The default resampling multiplier is `2`. This creates 75,448 GEE training
rows from the formal 37,724 rows. If the later GEE-vs-sklearn agreement is
below the operational threshold, increase it to `3` or `4`, rerun this
notebook, re-upload the three tables, and repeat the agreement check.
`gee_cost` is adjusted automatically, so do not change `C=10` manually.


In [ ]:
# ---------------------------------------------------------------------
# Confirmed Google Drive inputs
# ---------------------------------------------------------------------
INPUT_CSV = DATA_ROOT / r"Bentong_pixel_samples_2025_v1.csv"
BUNDLE_PATH = DATA_ROOT / r"SVM_grouped_validation_20260624_run01/models/svm_final_bundle.joblib"
TRAINING_ROWS_PATH = DATA_ROOT / r"RF_grouped_validation_20260623_153912_UTC/tables/training_rows_used.csv"

RUN_NAME = 'SVM_GEE_deployment_20260724_run01'
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME
TABLE_DIR = OUTPUT_DIR / 'tables_for_gee_upload'
METADATA_DIR = OUTPUT_DIR / 'metadata'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'B3', 'B4', 'B5', 'B6', 'B7',
    'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDRE', 'NDWI', 'EVI',
]
Z_FEATURES = [f'z_{name}' for name in FEATURES]

CLASS_TO_ID = {
    'Built-up/Bare soil': 0,
    'Durian': 1,
    'Forest': 2,
    'Mixed agriculture': 3,
    'Oil palm': 4,
    'Rubber': 5,
    'Water': 6,
}
ID_TO_CLASS = {value: key for key, value in CLASS_TO_ID.items()}
CLASS_IDS = sorted(ID_TO_CLASS)

EXPECTED_MODEL_ROWS = 37_724
EXPECTED_SAMPLE_COUNT = 557
EXPECTED_GROUP_COUNT = 206
EXPECTED_TRAINING_ROWS_SHA256 = (
    '0de3149cc9dd43fdccca6081db2d4939966896c585e5c61879740590987fdfef'
)
EXPECTED_INPUT_CSV_SHA256 = (
    '39bea3aca9b202e7e9155dbc0a68eb04795709db91ecda7edd2650489acdb0fb'
)

RANDOM_SEED = 42
CSV_CHUNK_SIZE = 100_000
VERIFICATION_PER_CLASS = 2_000
RESAMPLE_MULTIPLIER = 2
VERIFY_LARGE_INPUT_SHA256 = False

# Target GEE assets to create manually from the three output CSV files.
GEE_TRAINING_ASSET = (
    'projects/YOUR_GEE_PROJECT/assets/'
    'bentong_svm_s2_r1_deployment_training'
)
GEE_VERIFICATION_ASSET = (
    'projects/YOUR_GEE_PROJECT/assets/'
    'bentong_svm_s2_r1_deployment_verification'
)
GEE_CONFIG_ASSET = (
    'projects/YOUR_GEE_PROJECT/assets/'
    'bentong_svm_s2_r1_deployment_config'
)

if not isinstance(RESAMPLE_MULTIPLIER, int) or RESAMPLE_MULTIPLIER < 1:
    raise ValueError('RESAMPLE_MULTIPLIER must be a positive integer.')

missing_inputs = [
    str(path)
    for path in [INPUT_CSV, BUNDLE_PATH, TRAINING_ROWS_PATH]
    if not path.exists()
]
if missing_inputs:
    raise FileNotFoundError(
        'Missing confirmed input file(s):\n' + '\n'.join(missing_inputs)
    )

print('Output directory:', OUTPUT_DIR)
print('Resampling multiplier:', RESAMPLE_MULTIPLIER)


## 3. Reusable audit and atomic-output functions

Atomic writing prevents a disconnected Colab session from leaving a
partially written file under the final filename.


In [ ]:
def sha256_file(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            block = stream.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as stream:
        json.dump(payload, stream, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def assert_equal(left, right, message):
    if left != right:
        raise AssertionError(f'{message}: {left!r} != {right!r}')


def systematic_resample(probabilities, output_size, random_seed):
    # Low-variance deterministic resampling with one seeded offset.
    probabilities = np.asarray(probabilities, dtype='float64')
    if probabilities.ndim != 1 or len(probabilities) == 0:
        raise ValueError('probabilities must be a non-empty 1-D array.')
    if np.any(probabilities < 0) or not np.isfinite(probabilities).all():
        raise ValueError('probabilities contain invalid values.')
    probabilities = probabilities / probabilities.sum()
    cumulative = np.cumsum(probabilities)
    cumulative[-1] = 1.0
    rng = np.random.default_rng(random_seed)
    positions = (np.arange(output_size) + rng.random()) / output_size
    return np.searchsorted(cumulative, positions, side='right')


## 4. Load and audit the frozen sklearn bundle

This extracts the actual fitted scaler and the numeric value behind
`gamma='scale'`. No parameter is inferred from memory or re-estimated.


In [ ]:
bundle_sha256 = sha256_file(BUNDLE_PATH)
bundle = joblib.load(BUNDLE_PATH)

required_bundle_keys = {
    'model', 'feature_set', 'predictor_bands', 'class_to_id',
    'svm_params', 'candidate_id', 'random_seed',
}
missing_bundle_keys = required_bundle_keys.difference(bundle)
if missing_bundle_keys:
    raise KeyError(f'Frozen bundle is missing: {sorted(missing_bundle_keys)}')

frozen_model = bundle['model']
assert_equal(bundle['feature_set'], 'S2', 'Unexpected feature set')
assert_equal(bundle['candidate_id'], 'R1', 'Unexpected SVM candidate')
assert_equal(
    list(bundle['predictor_bands']),
    FEATURES,
    'Frozen feature order mismatch',
)
assert_equal(
    {str(k): int(v) for k, v in bundle['class_to_id'].items()},
    CLASS_TO_ID,
    'Class mapping mismatch',
)
assert_equal(
    list(frozen_model.named_steps),
    ['scaler', 'classifier'],
    'Unexpected pipeline steps',
)

scaler = frozen_model.named_steps['scaler']
classifier = frozen_model.named_steps['classifier']

scaler_mean = np.asarray(scaler.mean_, dtype='float64')
scaler_scale = np.asarray(scaler.scale_, dtype='float64')
if len(scaler_mean) != len(FEATURES):
    raise AssertionError('Scaler feature count is not 13.')
if not np.isfinite(scaler_mean).all():
    raise AssertionError('Scaler means contain non-finite values.')
if not np.isfinite(scaler_scale).all() or np.any(scaler_scale <= 0):
    raise AssertionError('Scaler scales are invalid.')

assert_equal(classifier.kernel, 'rbf', 'Unexpected SVC kernel')
assert_equal(float(classifier.C), 10.0, 'Unexpected frozen C')
assert_equal(float(classifier.tol), 0.001, 'Unexpected frozen tolerance')
assert_equal(
    classifier.classes_.astype(int).tolist(),
    CLASS_IDS,
    'SVC class order mismatch',
)
if bool(classifier.probability):
    raise AssertionError('Frozen SVC unexpectedly uses probability=True.')

effective_gamma = float(classifier._gamma)
frozen_cost = float(classifier.C)
gee_cost = frozen_cost / float(RESAMPLE_MULTIPLIER)
support_vector_count = int(classifier.support_vectors_.shape[0])

model_audit = pd.DataFrame({
    'feature': FEATURES,
    'weighted_scaler_mean': scaler_mean,
    'weighted_scaler_scale': scaler_scale,
})
display(model_audit)
print('Frozen bundle SHA256:', bundle_sha256)
print('Effective numeric gamma:', effective_gamma)
print('Frozen sklearn C:', frozen_cost)
print('GEE compensated cost:', gee_cost)
print('Support vectors:', f'{support_vector_count:,}')
print('Support vectors per class:', classifier.n_support_.tolist())
print('break_ties:', bool(getattr(classifier, 'break_ties', False)))


## 5. Load the exact formal training-row design

This reuses the same 37,724 `pixel_uid` values, 557 polygons, 206 spatial
groups and polygon-equalizing weights used in the completed Bentong model.


In [ ]:
training_rows_sha256 = sha256_file(TRAINING_ROWS_PATH)
if training_rows_sha256 != EXPECTED_TRAINING_ROWS_SHA256:
    raise ValueError(
        'training_rows_used.csv SHA256 differs from the frozen-model '
        'manifest. Stop before deployment.'
    )

training_rows = pd.read_csv(
    TRAINING_ROWS_PATH,
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
for column in ['class_id', 'fold_id']:
    training_rows[column] = pd.to_numeric(
        training_rows[column], errors='raise'
    ).astype(int)
training_rows['sample_weight'] = pd.to_numeric(
    training_rows['sample_weight'], errors='raise'
).astype('float64')

assert_equal(len(training_rows), EXPECTED_MODEL_ROWS, 'Model-row count')
assert_equal(
    training_rows['sample_uid'].nunique(),
    EXPECTED_SAMPLE_COUNT,
    'Polygon count',
)
assert_equal(
    training_rows['group_uid'].nunique(),
    EXPECTED_GROUP_COUNT,
    'Spatial-group count',
)
assert_equal(
    sorted(training_rows['class_id'].unique().tolist()),
    CLASS_IDS,
    'Training class IDs',
)
if training_rows['pixel_uid'].isna().any():
    raise ValueError('training_rows pixel_uid contains null values.')
if training_rows['pixel_uid'].duplicated().any():
    raise ValueError('training_rows pixel_uid is not unique.')
if (training_rows['sample_weight'] <= 0).any():
    raise ValueError('sample_weight must be positive.')

polygon_weight_totals = training_rows.groupby('sample_uid')[
    'sample_weight'
].sum()
if not np.allclose(polygon_weight_totals.to_numpy(), 1.0):
    raise ValueError('Per-polygon sample weights do not sum to one.')

design_summary = (
    training_rows.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygons=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
display(design_summary)
print('training_rows_used.csv SHA256:', training_rows_sha256)


## 6. Recover the 13 raw predictors for those exact rows

The original CSV is read in chunks so Colab does not need to hold the full
pixel table in memory. Class and polygon metadata are cross-checked after
the merge.


In [ ]:
required_source_columns = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'class_id',
    'longitude', 'latitude',
] + FEATURES
source_header = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
missing_source_columns = sorted(
    set(required_source_columns).difference(source_header)
)
if missing_source_columns:
    raise ValueError(
        f'Original Bentong CSV is missing: {missing_source_columns}'
    )

if VERIFY_LARGE_INPUT_SHA256:
    input_csv_sha256 = sha256_file(INPUT_CSV)
    if input_csv_sha256 != EXPECTED_INPUT_CSV_SHA256:
        raise ValueError('Original Bentong CSV SHA256 mismatch.')
else:
    input_csv_sha256 = None
    print(
        'Large input SHA256 check skipped. Set '
        'VERIFY_LARGE_INPUT_SHA256=True for a full byte-level audit.'
    )

selected_pixel_ids = set(training_rows['pixel_uid'].astype(str))
feature_parts = []
scanned_rows = 0
retained_rows = 0

for chunk_index, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=required_source_columns,
        dtype={
            'pixel_uid': 'string',
            'sample_uid': 'string',
            'group_uid': 'string',
            'class_lv2': 'string',
        },
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    scanned_rows += len(chunk)
    keep = chunk['pixel_uid'].astype(str).isin(selected_pixel_ids)
    retained = chunk.loc[keep].copy()
    if not retained.empty:
        feature_parts.append(retained)
        retained_rows += len(retained)
    print(
        f'Chunk {chunk_index}: scanned={scanned_rows:,}; '
        f'retained={retained_rows:,}'
    )
    del chunk, retained
    gc.collect()

if not feature_parts:
    raise ValueError('No formal training pixel_uid was found in the CSV.')
feature_rows = pd.concat(feature_parts, ignore_index=True)
feature_parts.clear()
gc.collect()

assert_equal(
    len(feature_rows), EXPECTED_MODEL_ROWS, 'Recovered feature-row count'
)
if feature_rows['pixel_uid'].duplicated().any():
    raise ValueError('Recovered raw predictors contain duplicate pixel_uid.')

model_df = training_rows.merge(
    feature_rows,
    on='pixel_uid',
    how='left',
    suffixes=('_design', '_source'),
    validate='one_to_one',
)
del feature_rows
gc.collect()

for column in ['sample_uid', 'group_uid', 'class_lv2']:
    left = model_df[f'{column}_design'].astype(str)
    right = model_df[f'{column}_source'].astype(str)
    if not left.equals(right):
        raise ValueError(f'Source/design mismatch: {column}')
    model_df[column] = model_df[f'{column}_design']

design_class = pd.to_numeric(
    model_df['class_id_design'], errors='raise'
).astype(int)
source_class = pd.to_numeric(
    model_df['class_id_source'], errors='raise'
).astype(int)
if not design_class.equals(source_class):
    raise ValueError('Source/design mismatch: class_id')
model_df['class_id'] = design_class

drop_suffix_columns = [
    name
    for name in model_df.columns
    if name.endswith('_design') or name.endswith('_source')
]
model_df = model_df.drop(columns=drop_suffix_columns)

for feature in FEATURES:
    model_df[feature] = pd.to_numeric(
        model_df[feature], errors='coerce'
    ).astype('float32')
model_df[FEATURES] = model_df[FEATURES].replace(
    [np.inf, -np.inf], np.nan
)
if model_df[FEATURES].isna().any(axis=1).any():
    raise ValueError('A formal training row has a missing predictor.')

for coordinate in ['longitude', 'latitude']:
    model_df[coordinate] = pd.to_numeric(
        model_df[coordinate], errors='coerce'
    )
if model_df[['longitude', 'latitude']].isna().any(axis=1).any():
    raise ValueError('A formal training row has a missing coordinate.')

print('Recovered exact formal model table:', model_df.shape)
display(model_df.head())


## 7. Create the weighted-resampling GEE training table

`sample_weight × class-frequency factor` exactly recreates the frozen
classifier-weight formula. Systematic resampling converts the continuous
weights to integer repetitions with lower variance than ordinary random
bootstrap sampling.

If the table is multiplied by `r`, duplicated rows would multiply the
effective penalty by `r`. The exported `gee_cost = frozen_C / r`
compensates for that change.


In [ ]:
class_row_counts = model_df['class_id'].value_counts().sort_index()
if set(class_row_counts.index.astype(int)) != set(CLASS_IDS):
    raise ValueError('The formal model table is missing a class.')

class_factors = {
    int(class_id): len(model_df) / (len(CLASS_IDS) * int(row_count))
    for class_id, row_count in class_row_counts.items()
}
classifier_weights = (
    model_df['sample_weight'].to_numpy(dtype='float64')
    * model_df['class_id'].map(class_factors).to_numpy(dtype='float64')
)
classifier_weights *= len(classifier_weights) / classifier_weights.sum()
if not np.isclose(classifier_weights.mean(), 1.0):
    raise AssertionError('Classifier weights are not normalized to mean 1.')

deployment_row_count = len(model_df) * RESAMPLE_MULTIPLIER
resampled_indices = systematic_resample(
    classifier_weights / classifier_weights.sum(),
    deployment_row_count,
    RANDOM_SEED,
)

deployment_training = model_df.iloc[resampled_indices].copy()
deployment_training['source_classifier_weight'] = (
    classifier_weights[resampled_indices]
)
deployment_training.insert(
    0,
    'deployment_row_id',
    [f'DEP_{index:07d}' for index in range(1, len(deployment_training) + 1)],
)

standardized = scaler.transform(
    deployment_training[FEATURES].astype('float32')
)
standardized_frame = pd.DataFrame(
    standardized,
    columns=Z_FEATURES,
    index=deployment_training.index,
)
for column in Z_FEATURES:
    deployment_training[column] = standardized_frame[column].to_numpy()

training_export_columns = [
    'deployment_row_id', 'pixel_uid', 'sample_uid', 'group_uid',
    'class_lv2', 'class_id', 'longitude', 'latitude',
    'sample_weight', 'source_classifier_weight',
] + Z_FEATURES
deployment_training = deployment_training[
    training_export_columns
].reset_index(drop=True)

if deployment_training['deployment_row_id'].duplicated().any():
    raise AssertionError('deployment_row_id is not unique.')
if deployment_training[Z_FEATURES].isna().any(axis=1).any():
    raise AssertionError('Standardized deployment predictors contain nulls.')

training_output_path = (
    TABLE_DIR / 'bentong_svm_s2_r1_deployment_training.csv'
)
atomic_write_csv(deployment_training, training_output_path)

resampling_summary = (
    deployment_training.groupby(['class_id', 'class_lv2'])
    .agg(
        deployment_rows=('deployment_row_id', 'size'),
        unique_source_pixels=('pixel_uid', 'nunique'),
        polygons=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
display(resampling_summary)
print('GEE training table:', training_output_path)
print('Rows:', f'{len(deployment_training):,}')

del standardized, standardized_frame
gc.collect()


## 8. Build an independent deployment-agreement verification table

The original CSV is scanned a second time. Pixels used in the formal
37,724-row fit are excluded first. A stable hash selects up to 2,000
pixels per class without loading all unused pixels into memory.

If a class has no such pixels (Rubber can have all available pixels in
the formal fit), the code uses a documented fallback hierarchy:

1. formal sklearn rows not selected into the resampled GEE training table;
2. remaining formal sklearn rows if the first fallback is insufficient.

`sklearn_pred` is the reference for later GEE parity testing. Accuracy
against `true_class_id` printed here is descriptive only because these
pixels can belong to polygons already represented in model development.


In [ ]:
verification_columns = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'class_id',
    'longitude', 'latitude',
] + FEATURES
best_by_class = {class_id: None for class_id in CLASS_IDS}
scanned_rows_verify = 0
eligible_rows_verify = 0

for chunk_index, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=verification_columns,
        dtype={
            'pixel_uid': 'string',
            'sample_uid': 'string',
            'group_uid': 'string',
            'class_lv2': 'string',
        },
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    scanned_rows_verify += len(chunk)
    chunk = chunk.loc[
        ~chunk['pixel_uid'].astype(str).isin(selected_pixel_ids)
    ].copy()
    if chunk.empty:
        continue

    chunk['class_id'] = pd.to_numeric(
        chunk['class_id'], errors='coerce'
    )
    for coordinate in ['longitude', 'latitude']:
        chunk[coordinate] = pd.to_numeric(
            chunk[coordinate], errors='coerce'
        )
    for feature in FEATURES:
        chunk[feature] = pd.to_numeric(chunk[feature], errors='coerce')
    chunk[FEATURES] = chunk[FEATURES].replace(
        [np.inf, -np.inf], np.nan
    )
    chunk = chunk.dropna(
        subset=['class_id', 'longitude', 'latitude'] + FEATURES
    )
    chunk['class_id'] = chunk['class_id'].astype(int)
    chunk = chunk.loc[chunk['class_id'].isin(CLASS_IDS)].copy()
    eligible_rows_verify += len(chunk)

    stable_key = (
        chunk['pixel_uid'].astype(str)
        + '|deployment_parity|'
        + str(RANDOM_SEED)
    )
    chunk['_stable_hash'] = pd.util.hash_pandas_object(
        stable_key, index=False
    ).astype('uint64').to_numpy()

    for class_id in CLASS_IDS:
        candidate = chunk.loc[chunk['class_id'] == class_id]
        if candidate.empty:
            continue
        candidate = candidate.nsmallest(
            VERIFICATION_PER_CLASS, '_stable_hash'
        )
        existing = best_by_class[class_id]
        if existing is not None:
            candidate = pd.concat(
                [existing, candidate], ignore_index=True
            ).nsmallest(VERIFICATION_PER_CLASS, '_stable_hash')
        best_by_class[class_id] = candidate.copy()

    print(
        f'Chunk {chunk_index}: scanned={scanned_rows_verify:,}; '
        f'eligible unused rows={eligible_rows_verify:,}'
    )
    del chunk
    gc.collect()

# ------------------------------------------------------------------
# Complete classes that do not have enough pixels outside the formal
# sklearn fit. This is valid for deployment parity, but the source must
# remain explicit and must not be reported as independent validation.
# ------------------------------------------------------------------
gee_training_source_ids = set(
    deployment_training['pixel_uid'].astype(str)
)

def add_stable_hash(frame, source_tag):
    frame = frame.copy()
    stable_key = (
        frame['pixel_uid'].astype(str)
        + '|'
        + source_tag
        + '|'
        + str(RANDOM_SEED)
    )
    frame['_stable_hash'] = pd.util.hash_pandas_object(
        stable_key, index=False
    ).astype('uint64').to_numpy()
    frame['verification_source'] = source_tag
    return frame


def append_nonempty_rows(existing, additional):
    # Preserve numeric dtypes when a class starts with no unused rows.
    if additional is None or additional.empty:
        return existing
    if existing is None or existing.empty:
        return additional.copy().reset_index(drop=True)
    return pd.concat(
        [existing, additional], ignore_index=True
    )


for class_id in CLASS_IDS:
    existing = best_by_class[class_id]
    if existing is None:
        existing = pd.DataFrame(columns=verification_columns)
    else:
        existing = existing.copy()
        existing['verification_source'] = (
            'unused_by_frozen_sklearn_fit'
        )

    selected_ids = set(existing['pixel_uid'].astype(str))
    rows_needed = max(0, VERIFICATION_PER_CLASS - len(existing))

    if rows_needed:
        # Preferred fallback: these rows were used by frozen sklearn,
        # but were not selected into the resampled GEE training table.
        fallback_pool = model_df.loc[
            (model_df['class_id'] == class_id)
            & ~model_df['pixel_uid'].astype(str).isin(
                gee_training_source_ids
            )
            & ~model_df['pixel_uid'].astype(str).isin(selected_ids),
            verification_columns,
        ].copy()
        fallback_pool = add_stable_hash(
            fallback_pool,
            'frozen_fit_but_unused_by_gee_replica',
        )
        fallback_take = fallback_pool.nsmallest(
            rows_needed, '_stable_hash'
        )
        existing = append_nonempty_rows(
            existing, fallback_take
        )
        selected_ids.update(
            fallback_take['pixel_uid'].astype(str)
        )
        rows_needed = max(
            0, VERIFICATION_PER_CLASS - len(existing)
        )

    if rows_needed:
        # Last-resort parity fallback. These are not independent rows
        # for either model, so they are explicitly marked.
        fallback_pool = model_df.loc[
            (model_df['class_id'] == class_id)
            & ~model_df['pixel_uid'].astype(str).isin(selected_ids),
            verification_columns,
        ].copy()
        fallback_pool = add_stable_hash(
            fallback_pool,
            'formal_training_fallback',
        )
        fallback_take = fallback_pool.nsmallest(
            rows_needed, '_stable_hash'
        )
        existing = append_nonempty_rows(
            existing, fallback_take
        )

    if existing.empty:
        raise ValueError(
            f'No verification candidate exists for class ID {class_id}.'
        )
    if len(existing) < VERIFICATION_PER_CLASS:
        warnings.warn(
            f'Class ID {class_id} has only {len(existing):,} '
            'verification rows; continuing with all available rows.',
            RuntimeWarning,
        )
    best_by_class[class_id] = existing

verification = pd.concat(
    [best_by_class[class_id] for class_id in CLASS_IDS],
    ignore_index=True,
)
verification['class_id'] = pd.to_numeric(
    verification['class_id'], errors='raise'
).astype('int64')
verification = verification.sort_values(
    ['class_id', '_stable_hash']
).reset_index(drop=True)
if verification['pixel_uid'].duplicated().any():
    raise ValueError('Verification pixel_uid is not unique.')

# Source-aware overlap audit. The old unconditional check is not valid
# after adding documented fallback rows.
verification_pixel_ids = verification['pixel_uid'].astype(str)
unused_source_mask = verification['verification_source'].eq(
    'unused_by_frozen_sklearn_fit'
)
unexpected_frozen_overlap = (
    unused_source_mask
    & verification_pixel_ids.isin(selected_pixel_ids)
)
if unexpected_frozen_overlap.any():
    raise ValueError(
        'A row marked unused_by_frozen_sklearn_fit overlaps the '
        'formal sklearn training rows.'
    )

gee_holdout_mask = verification['verification_source'].eq(
    'frozen_fit_but_unused_by_gee_replica'
)
unexpected_gee_overlap = (
    gee_holdout_mask
    & verification_pixel_ids.isin(gee_training_source_ids)
)
if unexpected_gee_overlap.any():
    raise ValueError(
        'A row marked frozen_fit_but_unused_by_gee_replica occurs '
        'in the resampled GEE training table.'
    )

allowed_sources = {
    'unused_by_frozen_sklearn_fit',
    'frozen_fit_but_unused_by_gee_replica',
    'formal_training_fallback',
}
observed_sources = set(
    verification['verification_source'].dropna().astype(str)
)
unexpected_sources = observed_sources.difference(allowed_sources)
if unexpected_sources:
    raise ValueError(
        f'Unexpected verification_source values: '
        f'{sorted(unexpected_sources)}'
    )

verification[FEATURES] = verification[FEATURES].astype('float32')
verification['sklearn_pred'] = frozen_model.predict(
    verification[FEATURES]
).astype('int64')
verification_standardized = scaler.transform(verification[FEATURES])
verification_z = pd.DataFrame(
    verification_standardized,
    columns=Z_FEATURES,
    index=verification.index,
)
for column in Z_FEATURES:
    verification[column] = verification_z[column].to_numpy()

verification = verification.rename(columns={'class_id': 'true_class_id'})
verification['true_class_id'] = pd.to_numeric(
    verification['true_class_id'], errors='raise'
).astype('int64')
verification['sklearn_pred'] = pd.to_numeric(
    verification['sklearn_pred'], errors='raise'
).astype('int64')
verification.insert(
    0,
    'verification_row_id',
    [f'VER_{index:06d}' for index in range(1, len(verification) + 1)],
)
verification_export_columns = [
    'verification_row_id', 'pixel_uid', 'sample_uid', 'group_uid',
    'class_lv2', 'true_class_id', 'sklearn_pred',
    'verification_source', 'longitude', 'latitude',
] + Z_FEATURES
verification = verification[
    verification_export_columns
].reset_index(drop=True)

verification_output_path = (
    TABLE_DIR / 'bentong_svm_s2_r1_deployment_verification.csv'
)
atomic_write_csv(verification, verification_output_path)

descriptive_accuracy = accuracy_score(
    verification['true_class_id'], verification['sklearn_pred']
)
descriptive_matrix = confusion_matrix(
    verification['true_class_id'],
    verification['sklearn_pred'],
    labels=CLASS_IDS,
)
print('Verification table:', verification_output_path)
print('Rows:', f'{len(verification):,}')
display(
    verification.groupby(
        ['true_class_id', 'class_lv2', 'verification_source']
    )
    .size()
    .rename('rows')
    .reset_index()
    .sort_values(['true_class_id', 'verification_source'])
)
print(
    'Descriptive frozen-model pixel accuracy '
    '(NOT a formal validation estimate):',
    f'{descriptive_accuracy:.4f}',
)
display(pd.DataFrame(
    descriptive_matrix,
    index=[f'true_{ID_TO_CLASS[i]}' for i in CLASS_IDS],
    columns=[f'pred_{ID_TO_CLASS[i]}' for i in CLASS_IDS],
))

del verification_standardized, verification_z
gc.collect()


## 9. Export scaler/SVM configuration and deployment manifest

The one-row CSV is read by the GEE script. The JSON manifest records the
source and output hashes for reproducibility.


In [ ]:
config_record = {
    'model_name': 'Bentong_frozen_SVM_S2_R1',
    'feature_set': 'S2',
    'feature_count': len(FEATURES),
    'feature_order': '|'.join(FEATURES),
    'class_ids': '|'.join(map(str, CLASS_IDS)),
    'class_mapping_json': json.dumps(
        CLASS_TO_ID, ensure_ascii=False, sort_keys=True
    ),
    'start_date': '2025-01-01',
    'end_date_exclusive': '2026-01-01',
    'svm_type': 'C_SVC',
    'kernel_type': 'RBF',
    'decision_procedure': 'Voting',
    'shrinking': 1,
    'frozen_cost': frozen_cost,
    'resample_multiplier': RESAMPLE_MULTIPLIER,
    'gee_cost': gee_cost,
    'effective_gamma': effective_gamma,
    'frozen_sklearn_tolerance': float(classifier.tol),
    'gee_c_svc_tolerance_note': (
        'GEE libsvm does not expose terminationEpsilon for C_SVC; '
        'the frozen tolerance is metadata only.'
    ),
    'sklearn_break_ties': int(
        bool(getattr(classifier, 'break_ties', False))
    ),
    'deployment_training_rows': len(deployment_training),
    'verification_rows': len(verification),
    'random_seed': RANDOM_SEED,
    'frozen_support_vectors': support_vector_count,
}
for index, feature in enumerate(FEATURES):
    config_record[f'mean_{feature}'] = float(scaler_mean[index])
    config_record[f'scale_{feature}'] = float(scaler_scale[index])

config_frame = pd.DataFrame([config_record])
config_output_path = (
    TABLE_DIR / 'bentong_svm_s2_r1_deployment_config.csv'
)
atomic_write_csv(config_frame, config_output_path)

output_hashes = {
    training_output_path.name: sha256_file(training_output_path),
    verification_output_path.name: sha256_file(
        verification_output_path
    ),
    config_output_path.name: sha256_file(config_output_path),
}
deployment_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': RUN_NAME,
    'purpose': (
        'Prepare a GEE deployment replica of the frozen Bentong '
        'SVM + S2 + R1 classifier.'
    ),
    'important_limitation': (
        'The GEE model approximates continuous sklearn sample weights '
        'by deterministic systematic resampling. It must pass the '
        'GEE-vs-sklearn agreement check before map export.'
    ),
    'formal_model_unchanged': True,
    'input_csv': str(INPUT_CSV),
    'input_csv_expected_sha256': EXPECTED_INPUT_CSV_SHA256,
    'input_csv_verified_sha256': input_csv_sha256,
    'frozen_bundle': str(BUNDLE_PATH),
    'frozen_bundle_sha256': bundle_sha256,
    'training_rows': str(TRAINING_ROWS_PATH),
    'training_rows_sha256': training_rows_sha256,
    'features': FEATURES,
    'class_to_id': CLASS_TO_ID,
    'model_rows': len(model_df),
    'sample_count': model_df['sample_uid'].nunique(),
    'group_count': model_df['group_uid'].nunique(),
    'resample_multiplier': RESAMPLE_MULTIPLIER,
    'deployment_training_rows': len(deployment_training),
    'verification_rows': len(verification),
    'effective_gamma': effective_gamma,
    'frozen_cost': frozen_cost,
    'gee_cost': gee_cost,
    'frozen_sklearn_tolerance': float(classifier.tol),
    'gee_c_svc_tolerance_exposed': False,
    'gee_target_assets': {
        'training': GEE_TRAINING_ASSET,
        'verification': GEE_VERIFICATION_ASSET,
        'config': GEE_CONFIG_ASSET,
    },
    'output_sha256': output_hashes,
    'software': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
manifest_path = METADATA_DIR / 'deployment_manifest.json'
atomic_write_json(deployment_manifest, manifest_path)

display(config_frame.T)
print('Configuration table:', config_output_path)
print('Manifest:', manifest_path)
print(json.dumps(output_hashes, indent=2))


## 10. Final audit and exact GEE upload instructions

Upload each CSV through the Earth Engine Assets tab as a table. When the
upload dialog offers longitude/latitude geometry fields, choose
`longitude` and `latitude`. Geometry is useful for inspection but the
classifier reads only the standardized `z_*` properties.

Use the three asset IDs printed below exactly. If an older deployment
version exists, delete or rename it deliberately in the Assets panel
before uploading the replacement; this notebook never modifies a GEE
asset.


In [ ]:
expected_training_columns = {
    'deployment_row_id', 'class_id', 'longitude', 'latitude',
    *Z_FEATURES,
}
expected_verification_columns = {
    'verification_row_id', 'true_class_id', 'sklearn_pred',
    'longitude', 'latitude', *Z_FEATURES,
}
expected_config_columns = {
    'effective_gamma', 'gee_cost', 'frozen_sklearn_tolerance',
    *[f'mean_{feature}' for feature in FEATURES],
    *[f'scale_{feature}' for feature in FEATURES],
}
if not expected_training_columns.issubset(deployment_training.columns):
    raise AssertionError('Training export columns are incomplete.')
if not expected_verification_columns.issubset(verification.columns):
    raise AssertionError('Verification export columns are incomplete.')
if not expected_config_columns.issubset(config_frame.columns):
    raise AssertionError('Config export columns are incomplete.')

upload_plan = pd.DataFrame([
    {
        'local_drive_csv': str(training_output_path),
        'target_gee_asset': GEE_TRAINING_ASSET,
        'role': 'GEE LibSVM deployment training',
    },
    {
        'local_drive_csv': str(verification_output_path),
        'target_gee_asset': GEE_VERIFICATION_ASSET,
        'role': 'GEE-vs-sklearn agreement check',
    },
    {
        'local_drive_csv': str(config_output_path),
        'target_gee_asset': GEE_CONFIG_ASSET,
        'role': 'Frozen scaler and numeric SVM settings',
    },
])
upload_plan_path = METADATA_DIR / 'gee_upload_plan.csv'
atomic_write_csv(upload_plan, upload_plan_path)
display(upload_plan)

print('\nAll local deployment-preparation checks passed.')
print(
    '\nNext: upload the three CSVs, open '
    '08_GEE_SVM_classification_export.js, keep '
    'enableClassificationExports=false, and run the agreement check.'
)
print(
    'Only after the Console reports acceptable agreement should you '
    'change enableClassificationExports to true and create the four '
    'export tasks (two Image Assets + two Drive GeoTIFFs).'
)
